In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f

spark=SparkSession.builder.appName("JoinPractice").getOrCreate()

data = [
    (1, None, "CEO"),
    (2, 1, "Manager A"),
    (3, 1, "Manager B"),
    (4, 2, "Employee X"),
    (5, 3, "Employee Y"),
    (6,None,"Sham")
]
columns = ["emp_id", "mng_id","ename"]
df = spark.createDataFrame(data,columns)
df.show()

1. **Employee and Manager Names:**  
   Write a PySpark query to create a DataFrame that lists each employee along with their manager's name.  
   Display columns: **employee**, **manager**.

2. **CEO-level Employees:**  
   Modify the code to find and display only the employee(s) who do not have a manager (CEO-level employees).  
   Display columns: **employee**, **manager**.

3. **Direct Reports to Manager A:**  
   Extend the code to find all employees who directly report to "Manager A."  
   Display columns: **empid**, **ename**, **mrgid**.

4. **Hierarchy Level:**  
   Write a query to determine the hierarchy level of each employee, where the CEO is level 1, direct reports to the CEO are level 2, and so on.  
   Display columns: **empid**, **ename**, **mrgid**, **level**.

In [0]:
#Write a PySpark query to create a DataFrame that lists each employee along with their manager's name. Display columns employee and manager.
from pyspark.sql import functions as f

result = df.alias("a").join(df.alias("b"), f.col("a.mng_id")==f.col("b.emp_id"), "left")\
    .select(f.col("a.ename").alias("EmployeeName"),
            (f.col("b.ename").alias("ManagerName"))
    ).show()

In [0]:
#Modify the code to find and display only the employee(s) who do not have a manager(CEO-level employees). Display columns employee and manager.
from pyspark.sql import functions as f
new = df.alias("a").join(df.alias("b"), f.col("a.mng_id")==f.col("b.emp_id"), "left")\
    .filter(f.col("b.ename").isNull())\
    .select(f.col("a.ename").alias("EmployeeName"),
            f.col("b.ename").alias("ManagerName"))
new.show()

In [0]:
#Extend the code to find all employees who directly report to "Manager A." Display columns empid, ename, and mrgid.
from pyspark.sql import functions as f

newdf = df.alias("a").join(df.alias("b"),f.col("a.mng_id")==f.col("b.emp_id"), "left" )\
    .filter(f.col("b.ename")=="Manager A")\
    .select(f.col("a.emp_id"),f.col("a.ename"),f.col("a.mng_id"))
newdf.show()

In [0]:
from pyspark.sql import functions as f
herarchi = df.alias("a").join(df.alias("b"),f.col("a.mng_id")==f.col("b.emp_id"), "left" ).show()

In [0]:
from pyspark.sql import functions as f

# Level 1 = CEO (employee with no manager)
hierarchy = df.filter(f.col("mng_id").isNull()) \
    .select(
        "emp_id",
        "ename",
        "mng_id",
        f.lit(1).alias("level")
    )

current = hierarchy

for i in range(2, 10):  # assuming max hierarchy depth < 10
    next_level = df.alias("e") \
        .join(
            current.alias("c"),
            f.col("e.mng_id") == f.col("c.emp_id"),
            "inner"
        ) \
        .select(
            f.col("e.emp_id"),
            f.col("e.ename"),
            f.col("e.mng_id"),
            f.lit(i).alias("level")
        )

    hierarchy = hierarchy.union(next_level)
    current = next_level

hierarchy.orderBy("level", "emp_id").show()